# GitHub Copilot Workshop · Microsoft Fabric

GitHub Copilot is the coding assistant; Fabric is the execution target. Synthetic manufacturing data only. See `docs/notebooks.md` before running. Use a disposable sandbox: data notebooks perform guarded full refreshes, not production ingestion.


In [ ]:
workspace_id = ""
lakehouse_id = ""
sandbox = False
approval = ""
batch_id = ""


In [ ]:
"""Fail-closed Fabric notebook authorization, also embedded by the builder."""

import re
from uuid import UUID


def require_uuid(value: str, label: str) -> str:
    if not isinstance(value, str):
        raise ValueError(f"{label} must be an explicit canonical UUID")
    try:
        parsed = UUID(value)
    except (ValueError, AttributeError) as error:
        raise ValueError(f"{label} must be an explicit canonical UUID") from error
    if str(parsed) != value or parsed.int == 0:
        raise ValueError(f"{label} must be a nonzero lowercase canonical UUID")
    return value


def assert_sandbox_context(runtime_context, *, workspace_id: str, lakehouse_id: str,
                           sandbox: bool, approval: str, batch_id: str) -> None:
    """Names, implicit current-workspace defaults, and truthy strings are not consent."""
    require_uuid(workspace_id, "workspace_id")
    require_uuid(lakehouse_id, "lakehouse_id")
    if sandbox is not True or approval != workspace_id:
        raise ValueError("Require sandbox=True and approval equal to the exact workspace UUID")
    if not isinstance(batch_id, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", batch_id):
        raise ValueError("batch_id must be a nonempty, path-safe run identifier (at most 80 characters)")
    try:
        current_workspace = runtime_context["currentWorkspaceId"]
        default_lakehouse = runtime_context["defaultLakehouseId"]
    except (KeyError, TypeError, AttributeError) as error:
        raise ValueError("Fabric runtime workspace and default lakehouse UUIDs are required") from error
    if current_workspace != workspace_id or default_lakehouse != lakehouse_id:
        raise ValueError("Fabric runtime UUIDs do not match the explicitly approved sandbox")
    attached_workspace = runtime_context.get("defaultLakehouseWorkspaceId")
    if attached_workspace and attached_workspace != workspace_id:
        raise ValueError("Cross-workspace default lakehouses are not supported by this workshop")


def assert_model_target(*, workspace_id: str, model_workspace_id: str,
                        expected_model_id: str) -> None:
    require_uuid(model_workspace_id, "model_workspace_id")
    require_uuid(expected_model_id, "expected_model_id")
    if model_workspace_id != workspace_id:
        raise ValueError("The model must be in the explicitly approved sandbox workspace")

def require_sandbox():
    assert_sandbox_context(
        notebookutils.runtime.context, workspace_id=workspace_id, lakehouse_id=lakehouse_id,
        sandbox=sandbox, approval=approval, batch_id=batch_id,
    )

require_sandbox()


In [ ]:
"""Embedded Spark helpers for this disposable, synthetic full-refresh path only."""

from collections.abc import Callable
from datetime import UTC, datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

RUN_STARTED = datetime.now(UTC)
RUN_OUTPUTS = {}
SYNTHETIC_TABLES = {
    "02_bronze_ingest": {
        f"bronze.{name}" for name in (
            "dim_asset", "dim_customer", "dim_date", "dim_defect_type", "dim_line", "dim_plant",
            "dim_product", "dim_shift", "fact_anomaly_event", "fact_customer_order",
            "fact_maintenance_event", "fact_production_run", "fact_quality_inspection",
        )
    },
    "03_silver_transform": {
        f"silver.{name}" for name in (
            "fact_production_run", "fact_quality_inspection", "fact_maintenance_event",
            "fact_customer_order", "dim_date", "dim_shift", "dim_line", "dim_asset",
            "dim_defect_type", "dq_rejects",
        )
    },
    "04_silver_dims_catchup": {"silver.dim_product", "silver.dim_customer", "silver.dim_plant"},
    "05_gold_star_schema": {
        f"gold.{name}" for name in (
            "dim_date", "dim_plant", "dim_line", "dim_asset", "dim_shift", "dim_product",
            "dim_customer", "dim_defect_type", "fact_production_run", "fact_quality_inspection",
            "fact_maintenance_event", "fact_customer_order", "agg_plant_month",
        )
    },
}


def configure_lineage(spark_session: SparkSession, guard: Callable[[], None], run_batch_id: str,
                      notebook_name: str, source_revision: str):
    """Bind the explicit notebook context after its parameter and UUID guard cells."""
    global spark, require_sandbox, batch_id, NOTEBOOK_NAME, SOURCE_REVISION
    guard()
    if notebook_name not in SYNTHETIC_TABLES:
        raise ValueError("Unknown synthetic notebook")
    spark, require_sandbox, batch_id = spark_session, guard, run_batch_id
    NOTEBOOK_NAME, SOURCE_REVISION = notebook_name, source_revision


def add_bronze_lineage(df):
    return (df.withColumn("_source_file", F.col("_metadata.file_path"))
            .withColumn("_ingest_timestamp", F.current_timestamp())
            .withColumn("_source_system", F.lit("synthetic-manufacturing-csv"))
            .withColumn("_batch_id", F.lit(batch_id)))


def add_silver_lineage(df):
    business_columns = sorted(column for column in df.columns if not column.startswith("_"))
    return (df.withColumn("_processed_timestamp", F.current_timestamp())
            .withColumn("_record_hash", F.sha2(F.to_json(F.struct(*[
                F.col(column) for column in business_columns
            ]), options={"ignoreNullFields": "false"}), 256))
            .withColumn("_batch_id", F.lit(batch_id)))


def write_synthetic(df, table):
    require_sandbox()
    if table not in SYNTHETIC_TABLES[NOTEBOOK_NAME]:
        raise ValueError("Output is outside this notebook's fixed synthetic table allowlist")
    if table.startswith("gold."):
        df = df.withColumn("_built_timestamp", F.current_timestamp()).withColumn("_batch_id", F.lit(batch_id))
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(table)
    rows = spark.table(table).count()
    RUN_OUTPUTS[table] = rows
    return rows


def emit_run_metadata():
    require_sandbox()
    if set(RUN_OUTPUTS) != SYNTHETIC_TABLES[NOTEBOOK_NAME]:
        raise ValueError("Not all expected outputs were written; do not mark this notebook successful")
    spark.sql("CREATE SCHEMA IF NOT EXISTS workshop")
    rows_written = sum(RUN_OUTPUTS.values())
    rows_quarantined = RUN_OUTPUTS.get("silver.dq_rejects", 0)
    if NOTEBOOK_NAME.startswith("02_"):
        rows_read = rows_written
    else:
        source_layer = "silver" if NOTEBOOK_NAME.startswith("05_") else "bronze"
        inputs = [table.split(".", 1)[1] for table in RUN_OUTPUTS
                  if not table.endswith(".dq_rejects") and not table.endswith(".agg_plant_month")]
        rows_read = sum(spark.table(f"{source_layer}.{table}").count() for table in inputs)
    metadata = [(
        batch_id, NOTEBOOK_NAME, NOTEBOOK_NAME.split("_")[1], "synthetic_manufacturing",
        RUN_STARTED, datetime.now(UTC), rows_read, rows_written, rows_quarantined,
        "succeeded", 1, SOURCE_REVISION,
    )]
    audit = spark.createDataFrame(metadata, """
        batch_id string, notebook string, layer string, domain string,
        started_at timestamp, ended_at timestamp, rows_read long, rows_written long, rows_quarantined long,
        status string, dictionary_version int, source_git_sha string
    """)
    audit.write.mode("append").format("delta").saveAsTable("workshop._pipeline_runs")
    print(f"Recorded successful run {batch_id}; stop the Spark session when finished.")

configure_lineage(spark, require_sandbox, batch_id, '02_bronze_ingest', 'd0d8c57a5dfc3d9eff8a2339c0266395367733df')


# Lab 2 · Bronze — land the raw plant files as Delta tables

**Bronze is a faithful copy of the source.** You read the 13 K-Corp plant CSV files exactly as they arrived and
save each one as a Delta table in the **`bronze`** schema of your Lakehouse. You change **nothing**. The files
have deliberate data-quality problems, and those are fixed in Silver (Lab 3), not here.

| Layer | Schema | What lands there | Rule of thumb |
|---|---|---|---|
| **Bronze** | `bronze` | Raw files as tables, every column a *string* | Keep everything, judge nothing |
| Silver | `silver` | Cleaned, typed, de-duplicated, conformed | Decide what is true |
| Gold | `gold` | Star schema and business KPIs | Make it easy to use |

> Attach a schema-enabled, disposable Fabric Lakehouse as default. Supply its UUID and the workspace UUID
> in the generated parameters cell. A matching display name or pin alone is not authorization.


## 0 · Setup

Create the `bronze` schema, and set the folder where Lab 1 uploaded the files.


In [ ]:
require_sandbox()
from pyspark.sql import functions as F

LANDING = "Files/landing/plant"  # manually stage the 13 synthetic plant CSVs here

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
print("✅ bronze schema ready")


## 1 · Land one file, step by step

Start with the biggest fact: **`FactProductionRun.csv`**, one row per production run on a plant line.

`inferSchema` is **off** on purpose, so every column arrives as a `string`. If Spark guessed the types here, a
single bad value such as `20250230` in a date column could silently turn into a null. Bronze keeps the raw text,
so nothing is lost before Silver has decided what to do with it.


In [ ]:
require_sandbox()
raw_runs = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{LANDING}/FactProductionRun.csv")
)

raw_runs.printSchema()
display(raw_runs.limit(10))


### Add mandatory lineage, then save as a Delta table

Every Bronze table gets four extra columns so you can trace a row back to its origin:

- `_source_file` records the file the row came from. `_metadata` is a hidden column Spark provides for file sources.
- `_ingest_timestamp` records when the row was loaded.
- `_source_system` identifies the synthetic CSV source.
- `_batch_id` records the explicitly supplied workshop run.

`saveAsTable("bronze.fact_production_run")` writes a **Delta** table into the `bronze` schema. Delta is the open
table format behind everything in OneLake, and it is what lets the SQL endpoint, Power BI and other engines read the
same data without copying it.


In [ ]:
require_sandbox()
bronze_runs = add_bronze_lineage(raw_runs)
write_synthetic(bronze_runs, "bronze.fact_production_run")
print("bronze.fact_production_run:", spark.table("bronze.fact_production_run").count(), "rows")


### Peek at the dirt, but don't fix it here

The query below uses `spark.sql` so the Python authorization guard also runs for this cell. You
should see the three real teams (`Team-A`, `Team-B`, `Team-C`) plus messy spellings such as `team-a`,
`TEAM-B ` and `Team C`, typed by hand on the shop floor. Silver will conform them.


In [ ]:
require_sandbox()
display(spark.sql("""
SELECT OperatorTeam, COUNT(*) AS runs
FROM bronze.fact_production_run
GROUP BY OperatorTeam
ORDER BY OperatorTeam
"""))


## 2 · Land the other twelve files with a loop

Now that you have seen each step, wrap them in a small function and run it for every CSV in the folder.
File names in `PascalCase` become `snake_case` table names, so `FactCustomerOrder.csv` lands as
`bronze.fact_customer_order`.


In [ ]:
require_sandbox()
import re


def to_snake(name: str) -> str:
    """FactCustomerOrder -> fact_customer_order"""
    return re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()


def land_csv(file_stem: str) -> tuple[str, int]:
    table = f"bronze.{to_snake(file_stem)}"
    df = (
        spark.read.option("header", True).option("inferSchema", False)
        .csv(f"{LANDING}/{file_stem}.csv")
    )
    return table, write_synthetic(add_bronze_lineage(df), table)


EXPECTED_BRONZE = {
    "DimAsset": 40, "DimCustomer": 36, "DimDate": 912, "DimDefectType": 12, "DimLine": 8, "DimPlant": 4,
    "DimProduct": 21, "DimShift": 3, "FactAnomalyEvent": 4, "FactCustomerOrder": 4560,
    "FactMaintenanceEvent": 270, "FactProductionRun": 3673, "FactQualityInspection": 2736,
}
csv_files = sorted(f.name[:-4] for f in notebookutils.fs.ls(LANDING) if f.name.endswith(".csv"))
if set(csv_files) != set(EXPECTED_BRONZE):
    raise ValueError("Stage exactly the 13 synthetic plant CSVs; unexpected or missing CSVs are rejected")
print(f"Found {len(csv_files)} CSV files in {LANDING}\n")

for stem in (name for name in csv_files if name != "FactProductionRun"):
    table, rows = land_csv(stem)
    print(f"{table:38s} {rows:>7,} rows")


## 3 · Verify: are all 13 Bronze tables there, with the right row counts?

The workshop data is fixed, so everyone should get exactly these numbers. If a line shows ❌, re-check Lab 1:
were all 13 files uploaded into `Files/landing/plant/`?


In [ ]:
require_sandbox()
problems, total = [], 0
for stem, expected in EXPECTED_BRONZE.items():
    table = f"bronze.{to_snake(stem)}"
    actual = spark.table(table).count() if spark.catalog.tableExists(table) else 0
    total += actual
    ok = actual == expected
    print(f"{'✅' if ok else '❌'} {table:38s} {actual:>7,} (expected {expected:,})")
    if not ok:
        problems.append(table)

assert not problems, f"Row counts differ for: {problems}"
print(f"\n🎉 Bronze complete: {len(EXPECTED_BRONZE)} tables, {total:,} rows. On to Lab 3!")


In [ ]:
require_sandbox()
emit_run_metadata()
